## Configuration Layer
This module defines the data source configuration and helper functions for loading/saving datasets.\
It supports two environments:
- **`unity_catalog`** — reads/writes from Databricks Unity Catalog tables (default)
- **`local_files`** — reads/writes from local Parquet files (for GitHub/local execution)

This notebook is called via `%run` from other notebooks.

In [0]:
# =============================================================================
# CONFIGURATION LAYER
# =============================================================================
# Set DATA_SOURCE to switch between environments:
#   - "unity_catalog" : reads from Databricks Unity Catalog tables
#   - "local_files"   : reads from local CSV/Parquet files (e.g., cloned from GitHub)
# =============================================================================

DATA_SOURCE = "unity_catalog"  # Change to "local_files" for GitHub/local execution

# --- Unity Catalog settings ---
CATALOG = "dev_forge_default"
SCHEMA = "mt_davide"

# --- Local file settings (relative to repo root) ---
LOCAL_DATA_DIR = "./data"  # Directory where exported data files are stored

In [0]:
# =============================================================================
# DATASET REGISTRY
# =============================================================================
# Convention-based: any dataset_key resolves automatically to:
#   UC:    {CATALOG}.{SCHEMA}.{dataset_key}
#   Local: {LOCAL_DATA_DIR}/{dataset_key}.parquet
#
# Use DATASET_CONFIG only for OVERRIDES where the table/file name
# differs from the logical key (e.g., legacy naming).
# =============================================================================

DATASET_CONFIG = {
    # Override entries (key != table name)
    "opportunities": {
        "uc_table": f"{CATALOG}.{SCHEMA}.opportunities_masked_rc",
        "local_file": f"{LOCAL_DATA_DIR}/opportunities_masked_rc.parquet",
    },
    "rc": {
        "uc_table": f"{CATALOG}.{SCHEMA}.rc_anonymized_masked",
        "local_file": f"{LOCAL_DATA_DIR}/rc_anonymized_masked.parquet",
    },
}


def resolve_dataset(dataset_key: str) -> dict:
    """
    Resolve a dataset key to its UC table path and local file path.
    If the key is registered in DATASET_CONFIG, use that override.
    Otherwise, derive paths from the naming convention.
    """
    if dataset_key in DATASET_CONFIG:
        return DATASET_CONFIG[dataset_key]
    # Convention-based resolution
    return {
        "uc_table": f"{CATALOG}.{SCHEMA}.{dataset_key}",
        "local_file": f"{LOCAL_DATA_DIR}/{dataset_key}.parquet",
    }

In [0]:
# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def load_dataset(dataset_key: str) -> "DataFrame":
    """
    Load a dataset based on the configured DATA_SOURCE.
    Resolves the key via DATASET_CONFIG (if registered) or naming convention.

    Parameters
    ----------
    dataset_key : str
        Logical name of the dataset. If not in DATASET_CONFIG,
        resolves to {CATALOG}.{SCHEMA}.{dataset_key} (UC)
        or {LOCAL_DATA_DIR}/{dataset_key}.parquet (local).

    Returns
    -------
    DataFrame
        Spark DataFrame (or pandas DataFrame if spark is unavailable).
    """
    config = resolve_dataset(dataset_key)

    if DATA_SOURCE == "unity_catalog":
        return spark.read.table(config["uc_table"])
    elif DATA_SOURCE == "local_files":
        import pandas as pd
        file_path = config["local_file"]
        if file_path.endswith(".parquet"):
            pdf = pd.read_parquet(file_path)
        elif file_path.endswith(".csv"):
            pdf = pd.read_csv(file_path)
        else:
            raise ValueError(f"Unsupported file format: {file_path}")
        try:
            return spark.createDataFrame(pdf)
        except NameError:
            return pdf
    else:
        raise ValueError(f"Unknown DATA_SOURCE: {DATA_SOURCE}. Use 'unity_catalog' or 'local_files'.")


def save_dataset(df, dataset_key: str):
    """
    Save a dataset based on the configured DATA_SOURCE.
    Resolves the key via DATASET_CONFIG (if registered) or naming convention.

    Parameters
    ----------
    df : DataFrame
        Spark or pandas DataFrame to save.
    dataset_key : str
        Logical name. Resolves automatically if not in DATASET_CONFIG.
    """
    config = resolve_dataset(dataset_key)

    if DATA_SOURCE == "unity_catalog":
        df.write \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(config["uc_table"])
        print(f"\u2713 Saved to UC: {config['uc_table']}")
    elif DATA_SOURCE == "local_files":
        import os
        os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
        file_path = config["local_file"]
        if hasattr(df, 'toPandas'):
            pdf = df.toPandas()
        else:
            pdf = df
        if file_path.endswith(".parquet"):
            pdf.to_parquet(file_path, index=False)
        elif file_path.endswith(".csv"):
            pdf.to_csv(file_path, index=False)
        print(f"\u2713 Saved locally: {file_path}")

In [0]:
# =============================================================================
# EXPORT UTILITY (run once to generate local files for GitHub)
# =============================================================================

def export_to_local(*dataset_keys: str):
    """
    Export specified UC tables as Parquet files for GitHub distribution.
    If no keys provided, exports all registered datasets.

    Parameters
    ----------
    *dataset_keys : str
        Dataset keys to export. If empty, exports all from DATASET_CONFIG.
    """
    import os
    os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

    keys = dataset_keys if dataset_keys else DATASET_CONFIG.keys()

    for key in keys:
        config = resolve_dataset(key)
        try:
            df = spark.read.table(config["uc_table"])
            local_path = config["local_file"]
            df.toPandas().to_parquet(local_path, index=False)
            print(f"\u2713 Exported {key}: {local_path} ({df.count()} rows)")
        except Exception as e:
            print(f"\u2717 Skipped {key}: {e}")

    print("\nDone! Add the 'data/' folder to your Git repository.")
    print("Then set DATA_SOURCE = 'local_files' in mt_src for GitHub users.")


# Uncomment to run:
# export_to_local()  # exports all registered datasets
# export_to_local("opportunities_aggregation", "rc_aggregation")  # export specific ones

In [0]:
print(f"\u2713 mt_src loaded | DATA_SOURCE = '{DATA_SOURCE}'")
print(f"  Catalog: {CATALOG}.{SCHEMA}")
print(f"  Registered overrides: {list(DATASET_CONFIG.keys())}")
print(f"  Convention: any key 'X' \u2192 {CATALOG}.{SCHEMA}.X (UC) or {LOCAL_DATA_DIR}/X.parquet (local)")
print(f"  Functions: load_dataset(), save_dataset(), export_to_local()")

✓ mt_src loaded | DATA_SOURCE = 'unity_catalog'
  Catalog: dev_forge_default.mt_davide
  Registered overrides: ['opportunities', 'rc']
  Convention: any key 'X' → dev_forge_default.mt_davide.X (UC) or ./data/X.parquet (local)
  Functions: load_dataset(), save_dataset(), export_to_local()
